# Rol 3. Analista de series de tiempo (Dulce)

**Task 1.3.** En este notebook se evalúa qué tanto se pueden creer las cifras del modelo. Se comparan las particiones aleatoria y temporal, se interpreta la autocorrelación del monto diario y se revisa la métrica de exactitud. Al final se redacta la sección 3 del memo.

Se agrega la raíz del repositorio al `sys.path` para importar los módulos de `src/`. Los datos del enunciado están en `src/case_data.py`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import case_data as cd

from src.metrics import accuracy_of_constant_classifier

## 1. Partición aleatoria vs temporal

In [2]:
drop = cd.AUC_RANDOM_SPLIT - cd.AUC_TEMPORAL_SPLIT
print(f"AUC partición aleatoria: {cd.AUC_RANDOM_SPLIT}")
print(f"AUC partición temporal:  {cd.AUC_TEMPORAL_SPLIT}")
print(f"Caída absoluta: {drop:.2f} | caída relativa: {drop / cd.AUC_RANDOM_SPLIT:.1%}")

AUC partición aleatoria: 0.87
AUC partición temporal:  0.81
Caída absoluta: 0.06 | caída relativa: 6.9%


### a. ¿Por qué la partición aleatoria infla el AUC?

Las secuencias de cada remitente se cortaron en ventanas de 10 transacciones. Con una partición aleatoria, ventanas del mismo remitente quedan en entrenamiento y en validación, y ventanas de meses posteriores pueden usarse para entrenar. El modelo reconoce remitentes y patrones que ya vio, en lugar de generalizar a casos nuevos. Esto es una fuga de información.

La partición temporal reproduce la situación real, porque el modelo se entrena con el pasado (enero a marzo) y predice el futuro (abril y mayo). Con ella el AUC baja de 0.87 a 0.81, una caída de 6.9%. **La gerencia debe usar 0.81.**

## 2. Autocorrelación y longitud de ventana

In [3]:
lags = [7, 15, 30]
print(f"Ventana mínima que contiene todos los rezagos: {max(lags)} días")

Ventana mínima que contiene todos los rezagos: 30 días


### b. Interpretación y ventana recomendada

Los picos de autocorrelación indican que el monto diario se repite con ciclos regulares.

- **7 días.** Ciclo semanal, por ejemplo envíos en fin de semana.
- **15 días.** Ciclo quincenal, asociado al pago de salarios por quincena.
- **30 días.** Ciclo mensual, asociado al pago mensual.

Se recomienda una ventana de **30 días**, que es la ventana mínima que contiene los tres ciclos. Una ventana de 10 transacciones puede cubrir menos de un mes en remitentes frecuentes, y entonces el modelo no vería el ciclo mensual y confundiría un patrón normal con una anomalía.

## 3. Exactitud de un sistema que nunca alerta

In [4]:
acc = accuracy_of_constant_classifier(cd.PREVALENCE)
missed = cd.DAILY_SENDERS * cd.PREVALENCE
print(f"Exactitud del sistema que nunca alerta: {acc:.1%}")
print(f"Casos diarios que deja sin detectar: {missed:.0f} (recall = 0)")

Exactitud del sistema que nunca alerta: 99.8%
Casos diarios que deja sin detectar: 12 (recall = 0)


### c. ¿Por qué la exactitud no sirve?

Un sistema que nunca alerta logra 99.8% de exactitud y, aun así, deja sin detectar los 12 casos diarios. Con una clase positiva tan pequeña, la exactitud premia acertar en los negativos y no mide lo que interesa.

Se proponen estas métricas.

- **Recall.** Fracción de casos de lavado que se detectan.
- **Precisión a prevalencia real.** Fracción de alertas que son casos confirmados, calculada con 0.2% de positivos.
- **PR-AUC (área bajo la curva precisión-recall).** Resume el modelo en todos los umbrales y, a diferencia del AUC ROC, es sensible a la clase minoritaria.
- **Alertas diarias.** Permite comparar cada umbral con la capacidad del equipo (Rol 7).

### d. Sección 3 del memo

> **3. Series de tiempo.** Las cifras de AUC de los Roles 1 y 2 están infladas por la partición aleatoria de ventanas, que mezcla ventanas del mismo remitente y de meses futuros entre entrenamiento y validación. Con partición temporal el AUC baja de 0.87 a 0.81, y esta es la cifra que debe usar la gerencia. Se recomienda una ventana de 30 días, que contiene los ciclos de 7, 15 y 30 días. La exactitud no debe reportarse, porque un sistema que nunca alerta logra 99.8%.